<a href="https://colab.research.google.com/github/GaborVxxx/ml_notes/blob/main/POS_home_cooking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:

import nltk
from nltk.corpus import brown

nltk.download('brown')
nltk.download('universal_tagset')

corpus = brown.tagged_sents(tagset='universal')
print(corpus)
print(len(corpus))

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


[[('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')], [('The', 'DET'), ('jury', 'NOUN'), ('further', 'ADV'), ('said', 'VERB'), ('in', 'ADP'), ('term-end', 'NOUN'), ('presentments', 'NOUN'), ('that', 'ADP'), ('the', 'DET'), ('City', 'NOUN'), ('Executive', 'ADJ'), ('Committee', 'NOUN'), (',', '.'), ('which', 'DET'), ('had', 'VERB'), ('over-all', 'ADJ'), ('charge', 'NOUN'), ('of', 'ADP'), ('the', 'DET'), ('election', 'NOUN'), (',', '.'), ('``', '.'), ('deserves', 'VERB'), ('the', 'DET'), ('praise', 'NOUN'), ('and', 'CONJ'), ('thanks', 'NOUN'), ('of', 'ADP'), ('the', 'DET'), ('City

In [51]:

# split the words for its place, and keep it in the same order
inputs = []
targets = []

for sentence_tag_pairs in corpus:
  tokens = []
  target = []
  for token, tag in sentence_tag_pairs:
    tokens.append(token)
    target.append(tag)
  inputs.append(token)
  targets.append(target)

print(len(inputs))
print(len(targets))

57340
57340


In [52]:
from sklearn.model_selection import train_test_split

# split the data and the lables
train_inputs, test_inputs, train_targets, test_targets = train_test_split(inputs, targets, test_size=0.2)
print(len(train_inputs))
print(len(test_inputs))
print(len(train_targets))
print(len(test_targets))

45872
11468
45872
11468


In [53]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Convert sentences to sequences

MAX_VOCAB_SIZE = None

# capitalization might be useful BUT need testing
should_lowercase = False
word_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    lower=should_lowercase,
    oov_token='UNK' # we need to keep unknown words in place, to make sure the tagse are aligne with tha data correctly
)

word_tokenizer.fit_on_texts(train_inputs) # use all text to map the words
train_inputs_int = word_tokenizer.texts_to_sequences(train_inputs)
test_inputs_int = word_tokenizer.texts_to_sequences(test_inputs)

In [54]:

# get word -> integer mapping
word2idx = word_tokenizer.word_index
print(word2idx)
V = len(word2idx)
print('Found %s unique tokens.' % V)

{'UNK': 1, "''": 2, 'Af': 3, 'Governor': 4, "'": 5, '9': 6, 'methods': 7, 'process': 8, '2': 9, '7': 10, 'the': 11, 'business': 12, 'Introduction': 13, 'needed': 14, 'facilities': 15, 'Washington': 16, 'shelter': 17, '3': 18, 'birth': 19, 'U': 20, 'Texas': 21, 'foods': 22, 'case': 23, 'publications': 24, '4': 25, '17': 26, 'there': 27, 'Discussion': 28, '1': 29, 'needs': 30, 'Thirty': 31, 'policies': 32, 'children': 33, 'material': 34, 'industry': 35, 'a': 36, 'N': 37, 'costs': 38, 'Contact': 39, 'self': 40, 'buildings': 41, 'assistance': 42, 'flux': 43, 'activities': 44, 'Sam': 45, 'muscle': 46, 'gains': 47, 'program': 48, 'Lauderdale': 49, 'type': 50, '8': 51, 'research': 52, 'loans': 53, 'place': 54, 'model': 55, 'power': 56, 'out': 57, 'programs': 58, 'war': 59, 'win': 60, 'materials': 61, 'at': 62, 'Insurance': 63, 'experiences': 64, 'Moscow': 65, '5': 66, 'Trenton': 67, 'lambs': 68, 'Results': 69, 'order': 70, 'five': 71, 'societies': 72, 'key': 73, 'frankfurters': 74, 'character

In [55]:

# flatten list of lists
def flatten(list_of_lists):
  return [item for sublist in list_of_lists for item in sublist]

In [56]:
# we work out the unique list of lables
all_train_targets = set(flatten(train_targets))
all_test_targets = set(flatten(test_targets))
print(all_train_targets)
print(all_test_targets)
all_train_targets == all_test_targets

{'ADP', 'NUM', 'PRON', 'VERB', 'X', 'ADJ', 'DET', 'CONJ', 'NOUN', '.', 'PRT', 'ADV'}
{'ADP', 'PRON', 'NUM', 'VERB', 'X', 'ADJ', 'DET', 'CONJ', 'NOUN', '.', 'PRT', 'ADV'}


True

In [57]:

# convert target to sequences
tag_tokenizer = Tokenizer()
tag_tokenizer.fit_on_texts(targets) # use all target
print(tag_tokenizer.word_index)
train_targets_int = tag_tokenizer.texts_to_sequences(train_targets)
test_targets_int = tag_tokenizer.texts_to_sequences(test_targets)

{'noun': 1, 'verb': 2, '.': 3, 'adp': 4, 'det': 5, 'adj': 6, 'adv': 7, 'pron': 8, 'conj': 9, 'prt': 10, 'num': 11, 'x': 12}
